In [ ]:
%%sql -r dataframe_1

USE DATABASE FOOD_DELIVERY_DB;

In [ ]:
%%sql -r dataframe_12
CREATE SCHEMA IF NOT EXISTS LANDING;
USE SCHEMA LANDING;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
TYPE = CSV
FIELD_DELIMITER = ','
SKIP_HEADER = 1
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
NULL_IF = ('NULL', 'null', '');

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE FILE FORMAT JSON_FORMAT
TYPE = JSON;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE FILE FORMAT PARQUET_FORMAT
TYPE = PARQUET;

In [ ]:
%%sql -r dataframe_5
SHOW FILE FORMATS;

In [ ]:
%%sql -r dataframe_13

CREATE OR REPLACE STAGE LANDING_STAGE
URL = 'azure://fdsmanasvi.blob.core.windows.net/landing'
CREDENTIALS = (
    AZURE_SAS_TOKEN = 'sp=rl&st=2026-05-15T05:36:50Z&se=2026-05-19T13:51:50Z&spr=https&sv=2025-11-05&sr=c&sig=mYhf9NwnsKhIchB5ZpWvYMOIEeZzxL%2FU%2BBXYYlAkwSs%3D'
);

In [ ]:
%%sql -r dataframe_14
LIST @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE;

In [ ]:
%%sql -r dataframe_6
USE DATABASE FOOD_DELIVERY_DB;
USE SCHEMA BRONZE;

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE BRONZE.CUSTOMERS_Details (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone_number STRING,
    date_of_birth DATE,
    gender STRING,
    address_line1 STRING,
    city STRING,
    state STRING,
    pincode STRING,
    signup_date DATE,
    customer_segment STRING,
    is_active BOOLEAN,
    last_order_date DATE,
    updated_at TIMESTAMP
);

In [ ]:
%%sql -r dataframe_11
COPY INTO CUSTOMERS_DETAILS
FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/customers_master.csv
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_15
SELECT * 
FROM CUSTOMERS_DETAILS
LIMIT 10;

In [ ]:
%%sql -r dataframe_16
DESC TABLE CUSTOMERS_DETAILS;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE DELIVERY_AGENTS_RAW (
    agent_id STRING,
    agent_name STRING,
    phone_number STRING,
    city STRING,
    vehicle_type STRING,
    joining_date DATE,
    agent_rating FLOAT,
    availability_status STRING,
    updated_at TIMESTAMP
);

In [ ]:
%%sql -r dataframe_9
COPY INTO DELIVERY_AGENTS_RAW
FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/delivery_agents_master.csv
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_10
DROP TABLE IF EXISTS BRONZE.CUSTOMERS_DETAILS;

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE TABLE BRONZE.CUSTOMERS_DETAILS(
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone_number STRING,
    date_of_birth DATE,
    gender STRING,
    address_line1 STRING,
    city STRING,
    state STRING,
    pincode STRING,
    signup_date DATE,
    customer_segment STRING,
    is_active BOOLEAN,
    last_order_date DATE,
    updated_at TIMESTAMP,

    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_18
COPY INTO BRONZE.CUSTOMERS_DETAILS
FROM (
    SELECT
        $1, $2, $3, $4, $5, $6, $7, $8,
        $9, $10, $11, $12, $13, $14, $15, $16,
        CURRENT_TIMESTAMP(),
        METADATA$FILENAME
    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/customers_master.csv
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_19
SELECT * 
FROM CUSTOMERS_DETAILS
LIMIT 10;

In [ ]:
%%sql -r dataframe_20
CREATE OR REPLACE TABLE BRONZE.DELIVERY_AGENT_DETAILS (
    agent_id STRING,
    agent_name STRING,
    phone_number STRING,
    city STRING,
    vehicle_type STRING,
    joining_date DATE,
    agent_rating FLOAT,
    availability_status STRING,
    updated_at TIMESTAMP,

    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_21
COPY INTO BRONZE.DELIVERY_AGENT_DETAILS
FROM (
    SELECT
        $1,$2,$3,$4,$5,$6,$7,$8,$9,
        CURRENT_TIMESTAMP(),
        METADATA$FILENAME
    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/delivery_agents_master.csv
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_22
select * from DELIVERY_AGENT_DETAILS LIMIT 10;

In [ ]:
%%sql -r dataframe_23
CREATE OR REPLACE TABLE BRONZE.PROMOTION_DETAILS (
    promo_code STRING,
    promo_type STRING,
    discount_value NUMBER,
    min_order_value NUMBER,
    start_date DATE,
    end_date DATE,
    is_active BOOLEAN,
    updated_at TIMESTAMP,

    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_24
COPY INTO BRONZE.PROMOTION_DETAILS
FROM (
    SELECT
        $1,$2,$3,$4,$5,$6,$7,$8,
        CURRENT_TIMESTAMP(),
        METADATA$FILENAME
    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/promotions_master.csv
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_25
CREATE OR REPLACE TABLE BRONZE.ORDER_DETAILS (
    order_id STRING,
    customer_id STRING,
    restaurant_id STRING,
    agent_id STRING,
    order_placed_at TIMESTAMP,
    order_accepted_at TIMESTAMP,
    order_delivered_at TIMESTAMP,
    order_status STRING,
    total_amount NUMBER(10,2),
    discount_amount NUMBER(10,2),
    delivery_fee NUMBER(10,2),
    tax_amount NUMBER(10,2),
    final_amount NUMBER(10,2),
    delivery_distance_km FLOAT,
    estimated_delivery_time NUMBER,
    actual_delivery_time NUMBER,
    delivery_city STRING,
    delivery_pincode STRING,
    order_source STRING,
    promo_code STRING,
    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_26
COPY INTO BRONZE.ORDER_DETAILS
FROM (
    SELECT
        $1,$2,$3,$4,$5,$6,$7,$8,$9,$10,
        $11,$12,$13,$14,$15,$16,$17,$18,$19,$20,
        CURRENT_TIMESTAMP(),
        METADATA$FILENAME
    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/orders_raw.csv
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_27
select * from  order_details limit 10

In [ ]:
%%sql -r dataframe_28
CREATE OR REPLACE TABLE BRONZE.ORDER_ITEM_DETAILS (
    order_item_id STRING,
    order_id STRING,
    item_name STRING,
    category STRING,
    quantity NUMBER,
    unit_price NUMBER(10,2),
    total_price NUMBER(10,2),
    is_veg BOOLEAN,
    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_29
COPY INTO BRONZE.ORDER_ITEM_DETAILS
FROM (
    SELECT
        $1,$2,$3,$4,$5,$6,$7,$8,
        CURRENT_TIMESTAMP(),
        METADATA$FILENAME
    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/order_items_raw.csv
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.CSV_FORMAT
);

In [ ]:
%%sql -r dataframe_30
CREATE OR REPLACE TABLE BRONZE.PAYMENT_DETAILS (
    payment_id STRING,
    order_id STRING,
    amount NUMBER(10,2),
    payment_method STRING,
    payment_gateway STRING,
    payment_status STRING,
    payment_timestamp TIMESTAMP,
    refund_status STRING,
    refund_amount NUMBER(10,2),
    card_last4 STRING,
    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_31
COPY INTO BRONZE.PAYMENT_DETAILS
FROM (
    SELECT
        $1:payment_id::STRING,
        $1:order_id::STRING,
        $1:amount::NUMBER(10,2),
        $1:payment_method::STRING,
        $1:payment_gateway::STRING,
        $1:payment_status::STRING,

        TO_TIMESTAMP($1:payment_timestamp::NUMBER / 1000),

        $1:refund_status::STRING,
        $1:refund_amount::NUMBER(10,2),
        $1:card_last4::STRING,

        CURRENT_TIMESTAMP(),
        METADATA$FILENAME

    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/payments.parquet
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.PARQUET_FORMAT
);

In [ ]:
%%sql -r dataframe_32
CREATE OR REPLACE TABLE BRONZE.RESTAURANT_CATALOG (
    restaurant_id STRING,
    restaurant_name STRING,
    cuisine_type STRING,
    city STRING,
    state STRING,
    pincode STRING,
    rating FLOAT,
    average_prep_time NUMBER,
    commission_rate NUMBER(10,2),
    opening_time STRING,
    closing_time STRING,
    is_active BOOLEAN,
    updated_at TIMESTAMP,
    ingestion_ts TIMESTAMP,
    source_file_name STRING
);

In [ ]:
%%sql -r dataframe_33
COPY INTO BRONZE.RESTAURANT_CATALOG
FROM (
    SELECT
        $1:restaurant_id::STRING,
        $1:restaurant_name::STRING,
        $1:cuisine_type::STRING,
        $1:city::STRING,
        $1:state::STRING,
        $1:pincode::STRING,
        $1:rating::FLOAT,
        $1:average_prep_time::NUMBER,
        $1:commission_rate::NUMBER(10,2),
        $1:opening_time::STRING,
        $1:closing_time::STRING,
        $1:is_active::BOOLEAN,
        $1:updated_at::TIMESTAMP,
        CURRENT_TIMESTAMP(),
        METADATA$FILENAME
    FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/restaurants_catalog.json
)
FILE_FORMAT = (
    FORMAT_NAME = FOOD_DELIVERY_DB.LANDING.JSON_FORMAT
);

In [ ]:
%%sql -r dataframe_34
-- Manual testing with bad records

INSERT INTO BRONZE.CUSTOMERS_DETAILS (
    customer_id, first_name, last_name, email, phone_number,
    date_of_birth, gender, address_line1, city, state, pincode,
    signup_date, customer_segment, is_active, last_order_date,
    updated_at
)
VALUES (
    'CUST_BAD_001',
    'Fake',
    'Email',
    'notanemail',
    '9999999999',
    '1998-05-10',
    'Female',
    'Test Address',
    'Jaipur',
    'Rajasthan',
    '302001',
    CURRENT_DATE(),
    'Regular',
    TRUE,
    CURRENT_DATE(),
    CURRENT_TIMESTAMP(),
    
);

In [ ]:
%%sql -r dataframe_36
INSERT INTO BRONZE.CUSTOMERS_DETAILS
VALUES (
    'CUST_BAD_002',
    'Future',
    'Person',
    'future@test.com',
    '8888888888',
    '2050-01-01',
    'Male',
    'Future Street',
    'Delhi',
    'Delhi',
    '110001',
    CURRENT_DATE(),
    'Premium',
    TRUE,
    CURRENT_DATE(),
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
);

In [ ]:
%%sql -r dataframe_37
INSERT INTO BRONZE.CUSTOMERS_DETAILS
VALUES (
    NULL,
    'Null',
    'ID',
    'null@test.com',
    '7777777777',
    '1990-05-10',
    'Female',
    'Null Street',
    'Mumbai',
    'Maharashtra',
    '400001',
    CURRENT_DATE(),
    'Regular',
    TRUE,
    CURRENT_DATE(),
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
);

In [ ]:
%%sql -r dataframe_38
INSERT INTO BRONZE.CUSTOMERS_DETAILS
SELECT *
FROM BRONZE.CUSTOMERS_DETAILS
LIMIT 1;

In [ ]:
%%sql -r dataframe_39
INSERT INTO BRONZE.CUSTOMERS_DETAILS
VALUES (
    ' CUST_BAD_005 ',
    '  Manii ',
    ' Sharma ',
    ' TEST@MAIL.COM ',
    '6666666666',
    '1999-01-01',
    'Female',
    ' Some Address ',
    ' Gurgaon ',
    ' Haryana ',
    '122001',
    CURRENT_DATE(),
    'Occasional',
    TRUE,
    CURRENT_DATE(),
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
);

In [ ]:
%%sql -r dataframe_40
SELECT *
FROM BRONZE.CUSTOMERS_DETAILS
WHERE source_file_name = 'manual_test';

In [ ]:
%%sql -r dataframe_41
INSERT INTO BRONZE.RESTAURANT_DETAILS(
    restaurant_id,
    restaurant_name,
    cuisine_type,
    city,
    state,
    pincode,
    rating,
    average_prep_time,
    commission_rate,
    opening_time,
    closing_time,
    is_active,
    updated_at,
    ingestion_ts,
    source_file_name
)
VALUES

-- invalid rating > 5
(
    'REST_BAD_001',
    'Bad Rating Cafe',
    'Chinese',
    'Jaipur',
    'Rajasthan',
    '302001',
    8.5,
    30,
    12.5,
    '09:00',
    '23:00',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- negative commission
(
    'REST_BAD_002',
    'Negative Commission House',
    'Italian',
    'Delhi',
    'Delhi',
    '110001',
    4.2,
    35,
    -15.0,
    '10:00',
    '22:00',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- null city
(
    'REST_BAD_003',
    'No City Diner',
    'North Indian',
    NULL,
    'Maharashtra',
    '400001',
    4.0,
    25,
    10.0,
    '08:00',
    '21:00',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- null restaurant id
(
    NULL,
    'Ghost Kitchen',
    'Fast Food',
    'Mumbai',
    'Maharashtra',
    '400002',
    3.8,
    20,
    9.5,
    '11:00',
    '23:30',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid prep time
(
    'REST_BAD_005',
    'Zero Prep Hub',
    'Mexican',
    'Gurgaon',
    'Haryana',
    '122001',
    4.1,
    0,
    8.0,
    '09:30',
    '22:30',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
);

In [ ]:
%%sql -r dataframe_42
INSERT INTO BRONZE.RESTAURANT_DETAILS
SELECT
    value:restaurant_id::STRING,
    value:restaurant_name::STRING,
    value:cuisine_type::STRING,
    value:city::STRING,
    value:state::STRING,
    value:pincode::STRING,
    value:rating::FLOAT,
    value:average_prep_time::NUMBER,
    value:commission_rate::NUMBER(10,2),
    value:opening_time::STRING,
    value:closing_time::STRING,
    value:is_active::BOOLEAN,
    value:updated_at::TIMESTAMP,
    CURRENT_TIMESTAMP(),
    METADATA$FILENAME
FROM @FOOD_DELIVERY_DB.LANDING.LANDING_STAGE/restaurants_catalog.json (FILE_FORMAT => 'FOOD_DELIVERY_DB.LANDING.JSON_FORMAT'),
     LATERAL FLATTEN(input => $1);

In [ ]:
%%sql -r dataframe_43
TRUNCATE TABLE BRONZE.RESTAURANT_DETAILS;

In [ ]:
%%sql -r dataframe_44
INSERT INTO BRONZE.DELIVERY_AGENT_DETAILS (
    AGENT_ID,
    AGENT_NAME,
    PHONE_NUMBER,
    CITY,
    VEHICLE_TYPE,
    JOINING_DATE,
    AGENT_RATING,
    AVAILABILITY_STATUS,
    UPDATED_AT,
    INGESTION_TS,
    SOURCE_FILE_NAME
)
VALUES

-- invalid rating > 5
(
    'AGENT_BAD_001',
    'Bad Rating',
    '9999999999',
    'Jaipur',
    'BIKE',
    '2025-01-01',
    8.5,
    'AVAILABLE',
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- future joining date
(
    'AGENT_BAD_002',
    'Future Joiner',
    '8888888888',
    'Delhi',
    'SCOOTER',
    '2050-01-01',
    4.5,
    'AVAILABLE',
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid phone number
(
    'AGENT_BAD_003',
    'Invalid Phone',
    '12AB567',
    'Mumbai',
    'BIKE',
    '2024-05-01',
    4.2,
    'AVAILABLE',
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- null agent id
(
    NULL,
    'Ghost Agent',
    '6666666666',
    'Gurgaon',
    'BIKE',
    '2024-06-01',
    3.9,
    'AVAILABLE',
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid vehicle type
(
    'AGENT_BAD_005',
    'Wrong Vehicle',
    '7777777777',
    'Pune',
    'TRUCK',
    '2024-01-01',
    4.1,
    'AVAILABLE',
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid availability status
(
    'AGENT_BAD_006',
    'Wrong Status',
    '8888877777',
    'Noida',
    'BIKE',
    '2024-03-01',
    4.0,
    'SLEEPING',
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
);

In [ ]:
%%sql -r dataframe_45
DESC TABLE BRONZE.DELIVERY_AGENT_DETAILS;

In [ ]:
%%sql -r dataframe_35
INSERT INTO FOOD_DELIVERY_DB.BRONZE.PROMOTION_DETAILS (
    PROMO_CODE,
    PROMO_TYPE,
    DISCOUNT_VALUE,
    MIN_ORDER_VALUE,
    START_DATE,
    END_DATE,
    IS_ACTIVE,
    UPDATED_AT,
    INGESTION_TS,
    SOURCE_FILE_NAME
)
VALUES

-- negative discount
(
    'BADPROMO001',
    'PERCENTAGE',
    -10,
    200,
    '2025-01-01',
    '2025-12-31',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- end date before start date
(
    'BADPROMO002',
    'FLAT',
    100,
    500,
    '2025-12-31',
    '2025-01-01',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- null promo code
(
    NULL,
    'PERCENTAGE',
    20,
    300,
    '2025-01-01',
    '2025-12-31',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid promo type
(
    'BADPROMO004',
    'FREE_GIFT',
    50,
    100,
    '2025-01-01',
    '2025-12-31',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- negative minimum order
(
    'BADPROMO005',
    'FLAT',
    100,
    -200,
    '2025-01-01',
    '2025-12-31',
    TRUE,
    CURRENT_TIMESTAMP(),
    CURRENT_TIMESTAMP(),
    'manual_test'
);

In [ ]:
%%sql -r dataframe_47
SELECT *
FROM BRONZE.PROMOTION_DETAILS
WHERE SOURCE_FILE_NAME = 'manual_test';

In [ ]:
%%sql -r dataframe_46
INSERT INTO BRONZE.ORDER_DETAILS (
    ORDER_ID, CUSTOMER_ID, RESTAURANT_ID, AGENT_ID, PROMO_CODE,
    TOTAL_AMOUNT, DISCOUNT_AMOUNT, TAX_AMOUNT, FINAL_AMOUNT, DELIVERY_FEE,
    DELIVERY_DISTANCE_KM, ESTIMATED_DELIVERY_TIME, ACTUAL_DELIVERY_TIME,
    ORDER_PLACED_AT, ORDER_ACCEPTED_AT, ORDER_DELIVERED_AT,
    ORDER_STATUS, ORDER_SOURCE, DELIVERY_CITY, DELIVERY_PINCODE,
    INGESTION_TS, SOURCE_FILE_NAME
) VALUES
-- 1. NULL ORDER_ID
(NULL, 'CUST_101', 'REST_201', 'AGT_301', 'NONE', 100.00, 10.00, 5.00, 115.00, 20.00, 2.0, 30, 35, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 35, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Mumbai', '400001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 2. NULL CUSTOMER_ID
('ORD_002', NULL, 'REST_202', 'AGT_302', 'NONE', 150.00, 0.00, 7.50, 172.50, 15.00, 1.5, 20, 22, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 22, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Delhi', '110001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 3. NULL RESTAURANT_ID
('ORD_003', 'CUST_103', NULL, 'AGT_303', 'WELCOME', 200.00, 50.00, 10.00, 180.00, 20.00, 4.0, 45, 40, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 40, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Bangalore', '560001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 4. Negative TOTAL_AMOUNT
('ORD_004', 'CUST_104', 'REST_204', 'AGT_304', 'NONE', -100.00, 0.00, 5.00, -75.00, 20.00, 3.2, 30, 28, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 28, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Pune', '411001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 5. Negative DISCOUNT_AMOUNT
('ORD_005', 'CUST_105', 'REST_205', 'AGT_305', 'BADPROMO', 300.00, -50.00, 15.00, 385.00, 20.00, 2.8, 35, 33, CURRENT_TIMESTAMP(), DATEADD(minute, 4, CURRENT_TIMESTAMP()), DATEADD(minute, 33, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Chennai', '600001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 6. Negative TAX_AMOUNT
('ORD_006', 'CUST_106', 'REST_206', 'AGT_306', 'NONE', 250.00, 0.00, -10.00, 260.00, 20.00, 5.1, 40, 42, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 42, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Hyderabad', '500001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 7. Negative FINAL_AMOUNT
('ORD_007', 'CUST_107', 'REST_207', 'AGT_307', 'NONE', 50.00, 100.00, 2.50, -27.50, 20.00, 1.0, 15, 12, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 12, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Kolkata', '700001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 8. Negative DELIVERY_FEE
('ORD_008', 'CUST_108', 'REST_208', 'AGT_308', 'NONE', 400.00, 0.00, 20.00, 400.00, -20.00, 6.5, 50, 48, CURRENT_TIMESTAMP(), DATEADD(minute, 5, CURRENT_TIMESTAMP()), DATEADD(minute, 48, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Jaipur', '302001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 9. Zero DELIVERY_DISTANCE_KM
('ORD_009', 'CUST_109', 'REST_209', 'AGT_309', 'NONE', 120.00, 0.00, 6.00, 156.00, 30.00, 0.0, 20, 18, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 18, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Ahmedabad', '380001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 10. Negative DELIVERY_DISTANCE_KM
('ORD_010', 'CUST_110', 'REST_210', 'AGT_310', 'NONE', 180.00, 0.00, 9.00, 209.00, 20.00, -2.5, 25, 26, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 26, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Surat', '395001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 11. ORDER_ACCEPTED_AT before ORDER_PLACED_AT
('ORD_011', 'CUST_111', 'REST_211', 'AGT_311', 'NONE', 220.00, 0.00, 11.00, 251.00, 20.00, 3.0, 30, 31, CURRENT_TIMESTAMP(), DATEADD(minute, -10, CURRENT_TIMESTAMP()), DATEADD(minute, 31, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Lucknow', '226001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 12. ORDER_DELIVERED_AT before ORDER_ACCEPTED_AT
('ORD_012', 'CUST_112', 'REST_212', 'AGT_312', 'NONE', 150.00, 0.00, 7.50, 177.50, 20.00, 2.2, 25, 22, CURRENT_TIMESTAMP(), DATEADD(minute, 30, CURRENT_TIMESTAMP()), DATEADD(minute, 15, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Kanpur', '208001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 13. ORDER_DELIVERED_AT before ORDER_PLACED_AT
('ORD_013', 'CUST_113', 'REST_213', 'AGT_313', 'NONE', 310.00, 0.00, 15.50, 345.50, 20.00, 4.5, 40, 38, CURRENT_TIMESTAMP(), DATEADD(minute, 5, CURRENT_TIMESTAMP()), DATEADD(minute, -20, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Nagpur', '440001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 14. Invalid ORDER_STATUS (FLYING)
('ORD_014', 'CUST_114', 'REST_214', 'AGT_314', 'NONE', 190.00, 0.00, 9.50, 219.50, 20.00, 1.8, 20, 19, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 19, CURRENT_TIMESTAMP()), 'FLYING', 'ANDROID', 'Indore', '452001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 15. Invalid ORDER_STATUS (LOST)
('ORD_015', 'CUST_115', 'REST_215', 'AGT_315', 'NONE', 450.00, 50.00, 22.50, 442.50, 20.00, 7.0, 55, NULL, CURRENT_TIMESTAMP(), DATEADD(minute, 5, CURRENT_TIMESTAMP()), NULL, 'LOST', 'WEB', 'Thane', '400601', CURRENT_TIMESTAMP(), 'manual_test'),

-- 16. Invalid ORDER_STATUS (MAGIC)
('ORD_016', 'CUST_116', 'REST_216', 'AGT_316', 'NONE', 110.00, 0.00, 5.50, 135.50, 20.00, 2.5, 25, 24, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 24, CURRENT_TIMESTAMP()), 'MAGIC', 'IOS', 'Bhopal', '462001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 17. Invalid ORDER_SOURCE
('ORD_017', 'CUST_117', 'REST_217', 'AGT_317', 'NONE', 275.00, 0.00, 13.75, 308.75, 20.00, 3.8, 35, 36, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 36, CURRENT_TIMESTAMP()), 'DELIVERED', 'PIGEON', 'Visakhapatnam', '530001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 18. Empty DELIVERY_CITY
('ORD_018', 'CUST_118', 'REST_218', 'AGT_318', 'NONE', 320.00, 0.00, 16.00, 356.00, 20.00, 4.2, 35, 34, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 34, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', '', '390001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 19. Invalid DELIVERY_PINCODE format
('ORD_019', 'CUST_119', 'REST_219', 'AGT_319', 'NONE', 140.00, 0.00, 7.00, 167.00, 20.00, 1.5, 20, 18, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 18, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Patna', 'ABCDEF', CURRENT_TIMESTAMP(), 'manual_test'),

-- 20. Extremely unrealistic delivery time values
('ORD_020', 'CUST_120', 'REST_220', 'AGT_320', 'NONE', 500.00, 0.00, 25.00, 545.00, 20.00, 5.5, 99999, 99999, CURRENT_TIMESTAMP(), DATEADD(minute, 4, CURRENT_TIMESTAMP()), DATEADD(minute, 99999, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Vadodara', '390001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 21. FINAL_AMOUNT math failure (Total - Discount + Tax + Fee != Final)
('ORD_021', 'CUST_121', 'REST_221', 'AGT_321', 'NONE', 200.00, 0.00, 10.00, 9999.99, 20.00, 3.0, 30, 28, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 28, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Ghaziabad', '201001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 22. CUSTOMER_ID fake/nonexistent reference
('ORD_022', 'FAKE_CUST_9999', 'REST_222', 'AGT_322', 'NONE', 160.00, 0.00, 8.00, 188.00, 20.00, 2.4, 25, 26, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 26, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Ludhiana', '141001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 23. RESTAURANT_ID fake/nonexistent reference
('ORD_023', 'CUST_123', 'GHOST_KITCHEN_000', 'AGT_323', 'NONE', 330.00, 0.00, 16.50, 366.50, 20.00, 4.0, 35, 35, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 35, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Agra', '282001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 24. AGENT_ID fake/nonexistent reference
('ORD_024', 'CUST_124', 'REST_224', 'AGENT_007_BOND', 'NONE', 210.00, 0.00, 10.50, 240.50, 20.00, 2.9, 30, 29, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 29, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Nashik', '422001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 25. PROMO_CODE fake/nonexistent reference
('ORD_025', 'CUST_125', 'REST_225', 'AGT_325', 'INVALID_PROMO_XYZ', 400.00, 200.00, 20.00, 240.00, 20.00, 5.0, 40, 42, CURRENT_TIMESTAMP(), DATEADD(minute, 4, CURRENT_TIMESTAMP()), DATEADD(minute, 42, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Faridabad', '121001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 26. Duplicate ORDER_ID (Record 1)
('ORD_DUP_999', 'CUST_126', 'REST_226', 'AGT_326', 'NONE', 150.00, 0.00, 7.50, 177.50, 20.00, 1.8, 20, 20, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 20, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Meerut', '250001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 27. Duplicate ORDER_ID (Record 2)
('ORD_DUP_999', 'CUST_127', 'REST_227', 'AGT_327', 'NONE', 280.00, 0.00, 14.00, 314.00, 20.00, 3.5, 35, 32, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 32, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Rajkot', '360001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 28. ORDER_STATUS = DELIVERED but ORDER_DELIVERED_AT is NULL
('ORD_028', 'CUST_128', 'REST_228', 'AGT_328', 'NONE', 190.00, 0.00, 9.50, 219.50, 20.00, 2.5, 25, NULL, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), NULL, 'DELIVERED', 'WEB', 'Kalyan', '421301', CURRENT_TIMESTAMP(), 'manual_test'),

-- 29. ORDER_STATUS = CANCELLED but delivery timestamps still populated
('ORD_029', 'CUST_129', 'REST_229', 'AGT_329', 'NONE', 350.00, 0.00, 17.50, 387.50, 20.00, 4.8, 45, 40, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 40, CURRENT_TIMESTAMP()), 'CANCELLED', 'IOS', 'Vasai', '401202', CURRENT_TIMESTAMP(), 'manual_test'),

-- 30. ORDER_STATUS = PLACED but accepted timestamp already populated
('ORD_030', 'CUST_130', 'REST_230', NULL, 'NONE', 130.00, 0.00, 6.50, 156.50, 20.00, 1.5, 20, NULL, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), NULL, 'PLACED', 'ANDROID', 'Varanasi', '221001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 31. ORDER_STATUS = PICKED_UP but no agent assigned (NULL AGENT_ID)
('ORD_031', 'CUST_131', 'REST_231', NULL, 'NONE', 260.00, 0.00, 13.00, 293.00, 20.00, 3.2, 35, NULL, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), NULL, 'PICKED_UP', 'WEB', 'Srinagar', '190001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 32. ORDER_SOURCE lowercase values
('ORD_032', 'CUST_132', 'REST_232', 'AGT_332', 'NONE', 170.00, 0.00, 8.50, 198.50, 20.00, 2.0, 25, 24, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 24, CURRENT_TIMESTAMP()), 'DELIVERED', 'ios', 'Aurangabad', '431001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 33. ORDER_SOURCE malformed values
('ORD_033', 'CUST_133', 'REST_233', 'AGT_333', 'NONE', 420.00, 0.00, 21.00, 461.00, 20.00, 6.0, 50, 48, CURRENT_TIMESTAMP(), DATEADD(minute, 4, CURRENT_TIMESTAMP()), DATEADD(minute, 48, CURRENT_TIMESTAMP()), 'DELIVERED', '  Android App  ', 'Dhanbad', '826001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 34. Mixed multiple: Negative amount + NULL Customer
('ORD_034', NULL, 'REST_234', 'AGT_334', 'NONE', -50.00, 0.00, 2.50, -27.50, 20.00, 1.2, 15, 16, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 16, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Amritsar', '143001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 35. Mixed multiple: Empty city + Invalid Pincode
('ORD_035', 'CUST_135', 'REST_235', 'AGT_335', 'NONE', 240.00, 0.00, 12.00, 272.00, 20.00, 3.6, 35, 33, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 33, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', '   ', 'PIN123', CURRENT_TIMESTAMP(), 'manual_test'),

-- 36. Mixed multiple: Fly status + Time travel timestamps
('ORD_036', 'CUST_136', 'REST_236', 'AGT_336', 'NONE', 380.00, 0.00, 19.00, 419.00, 20.00, 5.2, 45, 42, CURRENT_TIMESTAMP(), DATEADD(minute, 50, CURRENT_TIMESTAMP()), DATEADD(minute, 5, CURRENT_TIMESTAMP()), 'FLYING', 'ANDROID', 'Navi Mumbai', '400703', CURRENT_TIMESTAMP(), 'manual_test'),

-- 37. Mixed multiple: Math failure + Zero distance
('ORD_037', 'CUST_137', 'REST_237', 'AGT_337', 'NONE', 150.00, 20.00, 7.50, 0.00, 20.00, 0.0, 20, 18, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 18, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Allahabad', '211001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 38. Mixed multiple: Null Order ID + Cancelled with delivery time
(NULL, 'CUST_138', 'REST_238', 'AGT_338', 'NONE', 290.00, 0.00, 14.50, 324.50, 20.00, 3.9, 40, 35, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 35, CURRENT_TIMESTAMP()), 'CANCELLED', 'IOS', 'Howrah', '711101', CURRENT_TIMESTAMP(), 'manual_test'),

-- 39. Math failure (huge final amount injection)
('ORD_039', 'CUST_139', 'REST_239', 'AGT_339', 'NONE', 100.00, 0.00, 5.00, 999999.99, 20.00, 1.5, 20, 19, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 19, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Ranchi', '834001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 40. Unrealistic delivery time (-5 mins)
('ORD_040', 'CUST_140', 'REST_240', 'AGT_340', 'NONE', 220.00, 0.00, 11.00, 251.00, 20.00, 2.8, -5, -5, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, -5, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Gwalior', '474001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 41. Status mismatch: DELIVERED but actual_delivery_time is NULL
('ORD_041', 'CUST_141', 'REST_241', 'AGT_341', 'NONE', 340.00, 0.00, 17.00, 377.00, 20.00, 4.4, 40, NULL, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 40, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Jabalpur', '482001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 42. Status mismatch: PLACED but delivered timestamp populated
('ORD_042', 'CUST_142', 'REST_242', NULL, 'NONE', 180.00, 0.00, 9.00, 209.00, 20.00, 2.1, 25, 26, CURRENT_TIMESTAMP(), NULL, DATEADD(minute, 26, CURRENT_TIMESTAMP()), 'PLACED', 'ANDROID', 'Coimbatore', '641001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 43. Mixed multiple: Delivery fee negative + math failure
('ORD_043', 'CUST_143', 'REST_243', 'AGT_343', 'NONE', 410.00, 0.00, 20.50, 500.00, -50.00, 5.8, 50, 45, CURRENT_TIMESTAMP(), DATEADD(minute, 5, CURRENT_TIMESTAMP()), DATEADD(minute, 45, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Vijayawada', '520001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 44. Mixed multiple: Tax amount negative + promo code applied incorrectly
('ORD_044', 'CUST_144', 'REST_244', 'AGT_344', 'ADD50', 250.00, 0.00, -25.00, 300.00, 20.00, 3.3, 35, 34, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 34, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Jodhpur', '342001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 45. Mixed multiple: Empty city + NULL restaurant
('ORD_045', 'CUST_145', NULL, 'AGT_345', 'NONE', 160.00, 0.00, 8.00, 188.00, 20.00, 1.9, 20, 21, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 21, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', NULL, '625001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 46. Mixed multiple: Accepted at before placed at + magic status
('ORD_046', 'CUST_146', 'REST_246', 'AGT_346', 'NONE', 300.00, 0.00, 15.00, 335.00, 20.00, 4.1, 40, 38, CURRENT_TIMESTAMP(), DATEADD(minute, -15, CURRENT_TIMESTAMP()), DATEADD(minute, 38, CURRENT_TIMESTAMP()), 'ABRACADABRA', 'WEB', 'Raipur', '492001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 47. Mixed multiple: Null customer + null restaurant + null agent
('ORD_047', NULL, NULL, NULL, 'NONE', 270.00, 0.00, 13.50, 303.50, 20.00, 3.7, 35, 36, CURRENT_TIMESTAMP(), DATEADD(minute, 3, CURRENT_TIMESTAMP()), DATEADD(minute, 36, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Kota', '324001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 48. Pincode with special characters
('ORD_048', 'CUST_148', 'REST_248', 'AGT_348', 'NONE', 190.00, 0.00, 9.50, 219.50, 20.00, 2.6, 30, 28, CURRENT_TIMESTAMP(), DATEADD(minute, 2, CURRENT_TIMESTAMP()), DATEADD(minute, 28, CURRENT_TIMESTAMP()), 'DELIVERED', 'ANDROID', 'Guwahati', '@#$%', CURRENT_TIMESTAMP(), 'manual_test'),

-- 49. Extremely unrealistic distance
('ORD_049', 'CUST_149', 'REST_249', 'AGT_349', 'NONE', 460.00, 0.00, 23.00, 503.00, 20.00, 99999.9, 55, 52, CURRENT_TIMESTAMP(), DATEADD(minute, 5, CURRENT_TIMESTAMP()), DATEADD(minute, 52, CURRENT_TIMESTAMP()), 'DELIVERED', 'WEB', 'Chandigarh', '160001', CURRENT_TIMESTAMP(), 'manual_test'),

-- 50. Nulls and Zeroes populated simultaneously across all numeric/string metrics
('ORD_050', 'CUST_150', 'REST_250', 'AGT_350', 'NONE', 0.00, 0.00, 0.00, 0.00, 0.00, 0.0, 0, 0, CURRENT_TIMESTAMP(), DATEADD(minute, 1, CURRENT_TIMESTAMP()), DATEADD(minute, 1, CURRENT_TIMESTAMP()), 'DELIVERED', 'IOS', 'Solapur', '413001', CURRENT_TIMESTAMP(), 'manual_test');

In [ ]:
%%sql -r dataframe_48
SELECT DISTINCT ORDER_STATUS
FROM BRONZE.ORDER_DETAILS;

In [ ]:
%%sql -r dataframe_49
SELECT DISTINCT ORDER_SOURCE
FROM BRONZE.ORDER_DETAILS;

In [ ]:
%%sql -r dataframe_50


In [ ]:
%%sql -r dataframe_51
INSERT INTO BRONZE.PAYMENT_DETAILS (
    PAYMENT_ID,
    ORDER_ID,
    AMOUNT,
    PAYMENT_METHOD,
    PAYMENT_GATEWAY,
    PAYMENT_STATUS,
    PAYMENT_TIMESTAMP,
    REFUND_STATUS,
    REFUND_AMOUNT,
    CARD_LAST4,
    INGESTION_TS,
    SOURCE_FILE_NAME
)
VALUES

-- null order id
(
    'PAY_BAD_001',
    NULL,
    500,
    'UPI',
    'RAZORPAY',
    'SUCCESS',
    CURRENT_TIMESTAMP(),
    'NONE',
    0,
    NULL,
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- fake order id
(
    'PAY_BAD_002',
    'FAKE_ORDER_001',
    1000,
    'CARD',
    'STRIPE',
    'SUCCESS',
    CURRENT_TIMESTAMP(),
    'NONE',
    0,
    '1234',
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- negative amount
(
    'PAY_BAD_003',
    'ORD0000001',
    -500,
    'UPI',
    'RAZORPAY',
    'SUCCESS',
    CURRENT_TIMESTAMP(),
    'NONE',
    0,
    NULL,
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid payment method
(
    'PAY_BAD_004',
    'ORD0000002',
    800,
    'BITCOIN',
    'CRYPTOPAY',
    'SUCCESS',
    CURRENT_TIMESTAMP(),
    'NONE',
    0,
    NULL,
    CURRENT_TIMESTAMP(),
    'manual_test'
),

-- invalid payment status
(
    'PAY_BAD_005',
    'ORD0000003',
    600,
    'CARD',
    'STRIPE',
    'MAGIC',
    CURRENT_TIMESTAMP(),
    'NONE',
    0,
    '5678',
    CURRENT_TIMESTAMP(),
    'manual_test'
);